# Week 3 Telemetry Pipeline Demo

This notebook demonstrates that the Week 3 telemetry pipeline works end to end:

1. generate a 10,000-row synthetic Aido Rover telemetry CSV  
2. ingest and validate schema  
3. clean GPS dropout, LiDAR saturation, and noise spikes  
4. extract telemetry features  
5. write Parquet + validation summary JSON  
6. run required self-checks  
7. profile runtime and identify the slowest stage  

Run this notebook from the repository root after installing dev dependencies:

```bash
python -m pip install -e ".[dev]"
```


In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ingen_pydev.pipeline.clean import clean_telemetry
from ingen_pydev.pipeline.features import add_telemetry_features
from ingen_pydev.pipeline.ingest import REQUIRED_COLUMNS, read_telemetry_csv
from ingen_pydev.pipeline.output import build_validation_summary, write_pipeline_outputs
from ingen_pydev.pipeline.run_pipeline import profile_pipeline, run_pipeline
from ingen_pydev.pipeline.synthetic import generate_synthetic_telemetry

pd.set_option("display.max_columns", 80)

DATA_PATH = Path("synthetic_data.csv")
OUTPUT_DIR = Path("outputs")
PARQUET_PATH = OUTPUT_DIR / "demo_cleaned_features.parquet"
SUMMARY_PATH = OUTPUT_DIR / "demo_validation_summary.json"


## 1. Generate synthetic telemetry data

This creates the required `synthetic_data.csv` with 10,000 timestamped rows.


In [2]:
df_raw = generate_synthetic_telemetry(DATA_PATH, rows=10_000, seed=42)

print("Synthetic dataset written to:", DATA_PATH)
print("Shape:", df_raw.shape)
print("Required columns present:", set(REQUIRED_COLUMNS).issubset(df_raw.columns))

df_raw.head()


Synthetic dataset written to: synthetic_data.csv
Shape: (10000, 10)
Required columns present: True


,timestamp_ms,lat,lon,lidar_distance_m,battery_soc,wheel_torque_fl,wheel_torque_fr,wheel_torque_rl,wheel_torque_rr,ambient_temp_c
0,1735689600000,40.110601,-88.207299,32.512699,99.878629,11.550116,12.384840,12.391336,11.094791,22.230385
1,1735689601000,40.110598,-88.207297,36.356919,99.992512,11.372799,11.079932,11.846245,12.330427,22.231033
2,1735689602000,40.110600,-88.207301,34.953168,99.834836,12.170078,11.883941,12.767605,12.165985,22.154575
3,1735689603000,40.110603,-88.207302,34.160460,99.908757,12.082327,11.460337,11.501022,11.689612,22.598007
4,1735689604000,40.110597,-88.207300,34.704205,99.963496,10.895261,13.439367,11.167117,11.429683,21.872599


In [3]:
# Quick anomaly sanity checks in the generated raw data.
gps_dropout_rows = int((df_raw["lat"].isna() | df_raw["lon"].isna()).sum())
lidar_saturation_rows = int(
    ((df_raw["lidar_distance_m"] <= 0.0) | (df_raw["lidar_distance_m"] >= 200.0)).sum()
)
battery_sudden_drop_rows = int((df_raw["battery_soc"].diff() < -20.0).sum())

pd.DataFrame(
    [
        {"check": "rows", "value": len(df_raw)},
        {"check": "GPS dropout rows", "value": gps_dropout_rows},
        {"check": "LiDAR saturation rows", "value": lidar_saturation_rows},
        {"check": "battery sudden-drop rows", "value": battery_sudden_drop_rows},
    ]
)


,check,value
0,rows,10000
1,GPS dropout rows,200
2,LiDAR saturation rows,111
3,battery sudden-drop rows,297


## 2. Run the full pipeline

The full pipeline chains:

```text
ingest → clean → features → output
```

The assignment target is processing 10,000 rows in under 2 seconds.


In [4]:
start = time.perf_counter()

result = run_pipeline(
    input_csv=DATA_PATH,
    output_parquet=PARQUET_PATH,
    summary_json=SUMMARY_PATH,
    generate_if_missing=False,
)

measured_elapsed = time.perf_counter() - start

print("Rows processed:", result.rows_processed)
print("Pipeline elapsed seconds:", round(result.elapsed_seconds, 4))
print("Measured wrapper elapsed seconds:", round(measured_elapsed, 4))
print("Under 2 seconds:", result.elapsed_seconds < 2.0)
print("Stage times:", result.stage_times_seconds)
print("Validation summary:", result.validation_summary)


Rows processed: 10000
Pipeline elapsed seconds: 2.8378
Measured wrapper elapsed seconds: 2.8386
Under 2 seconds: False
Stage times: {'ingest': 0.021196300047449768, 'clean': 0.18198200000915676, 'features': 0.013152600033208728, 'output': 2.6214285000460222}
Validation summary: {'rows_processed': 10000, 'rows_flagged': 1184, 'fields_with_anomalies_detected': ['gps', 'lidar_distance_m', 'battery_soc', 'wheel_torque', 'ambient_temp_c']}


## 3. Inspect cleaned feature output

The output should be readable as Parquet and contain both cleaned fields and derived features.


In [5]:
df_out = pd.read_parquet(PARQUET_PATH)

print("Output Parquet:", PARQUET_PATH)
print("Shape:", df_out.shape)

selected_columns = [
    "timestamp_ms",
    "lat",
    "lon",
    "lidar_distance_m",
    "battery_soc",
    "battery_soc_roll_mean_50",
    "battery_soc_roll_std_50",
    "cumulative_distance_m",
    "wheel_imbalance_score",
    "lidar_saturation_rate",
    "composite_health_score",
]

df_out[selected_columns].head()


Output Parquet: outputs\demo_cleaned_features.parquet
Shape: (10000, 27)


,timestamp_ms,lat,lon,lidar_distance_m,battery_soc,battery_soc_roll_mean_50,battery_soc_roll_std_50,cumulative_distance_m,wheel_imbalance_score,lidar_saturation_rate,composite_health_score
0,1735689600000,40.110601,-88.207299,32.512699,99.878629,99.878629,0.000000,0.000000,0.109364,0.0,99.878629
1,1735689601000,40.110598,-88.207297,36.356919,99.992512,99.935570,0.056942,0.415903,0.107271,0.0,99.992512
2,1735689602000,40.110600,-88.207301,34.953168,99.834836,99.901992,0.066457,0.867297,0.072154,0.0,99.834836
3,1735689603000,40.110603,-88.207302,34.160460,99.908757,99.903683,0.057628,1.189387,0.053237,0.0,99.908757
4,1735689604000,40.110597,-88.207300,34.704205,99.963496,99.915646,0.056826,1.873738,0.216836,0.0,99.963496


In [6]:
with SUMMARY_PATH.open("r", encoding="utf-8") as file:
    summary = json.load(file)

summary


{'rows_processed': 10000,
 'rows_flagged': 1184,
 'fields_with_anomalies_detected': ['gps',
  'lidar_distance_m',
  'battery_soc',
  'wheel_torque',
  'ambient_temp_c']}

## 4. Required self-check: GPS dropout cleaning

Requirement:

- exactly 3 consecutive GPS dropouts should be forward-filled  
- the 4th consecutive dropout should be flagged as a long dropout  


In [7]:
gps_test = pd.DataFrame(
    {
        "timestamp_ms": [1, 2, 3, 4, 5, 6],
        "lat": [40.0, np.nan, np.nan, np.nan, np.nan, 40.0005],
        "lon": [-88.0, np.nan, np.nan, np.nan, np.nan, -88.0005],
        "lidar_distance_m": [10.0, 11.0, 12.0, 13.0, 14.0, 15.0],
        "battery_soc": [90.0, 89.9, 89.8, 89.7, 89.6, 89.5],
        "wheel_torque_fl": [10.0] * 6,
        "wheel_torque_fr": [10.0] * 6,
        "wheel_torque_rl": [10.0] * 6,
        "wheel_torque_rr": [10.0] * 6,
        "ambient_temp_c": [22.0] * 6,
    }
)

cleaned_gps_test = clean_telemetry(gps_test)

check_columns = ["timestamp_ms", "lat", "lon", "gps_dropout", "gps_filled", "gps_dropout_long"]
cleaned_gps_test[check_columns]


,timestamp_ms,lat,lon,gps_dropout,gps_filled,gps_dropout_long
0,1,40.0000,-88.0000,False,False,False
1,2,40.0000,-88.0000,True,True,False
2,3,40.0000,-88.0000,True,True,False
3,4,40.0000,-88.0000,True,True,False
4,5,NaN,NaN,True,False,True
5,6,40.0005,-88.0005,False,False,False


In [8]:
# Rows 1, 2, 3 are the first three missing GPS rows and should be filled.
assert cleaned_gps_test.loc[[1, 2, 3], "gps_filled"].tolist() == [True, True, True]

# Row 4 is the 4th consecutive missing GPS row and should be flagged as long dropout.
assert bool(cleaned_gps_test.loc[4, "gps_dropout_long"]) is True

print("GPS dropout self-check passed.")


GPS dropout self-check passed.


## 5. Required self-check: rolling battery statistics

Requirement:

- `features.py` rolling mean/std must match pandas rolling ground truth.


In [9]:
feature_test = cleaned_gps_test.copy()
feature_out = add_telemetry_features(feature_test)

expected_mean = feature_test["battery_soc"].rolling(50, min_periods=1).mean()
expected_std = feature_test["battery_soc"].rolling(50, min_periods=1).std(ddof=0)

assert np.allclose(feature_out["battery_soc_roll_mean_50"], expected_mean)
assert np.allclose(feature_out["battery_soc_roll_std_50"], expected_std)

print("Rolling statistics self-check passed.")
feature_out[
    [
        "battery_soc",
        "battery_soc_roll_mean_50",
        "battery_soc_roll_std_50",
        "cumulative_distance_m",
        "wheel_imbalance_score",
        "composite_health_score",
    ]
]


Rolling statistics self-check passed.


,battery_soc,battery_soc_roll_mean_50,battery_soc_roll_std_50,cumulative_distance_m,wheel_imbalance_score,composite_health_score
0,90.0,90.00,0.000000,0.0,0.0,90.0
1,89.9,89.95,0.050000,0.0,0.0,89.9
2,89.8,89.90,0.081650,0.0,0.0,89.8
3,89.7,89.85,0.111803,0.0,0.0,89.7
4,89.6,89.80,0.141421,0.0,0.0,89.6
5,89.5,89.75,0.170783,0.0,0.0,89.5


## 6. Required self-check: Parquet round trip

Requirement:

- output Parquet file is readable  
- schema remains correct after reading it back  
- validation JSON counts flagged rows, not just processed rows  


In [10]:
required_output_columns = {
    "timestamp_ms",
    "lat",
    "lon",
    "lidar_distance_m",
    "battery_soc",
    "battery_soc_roll_mean_50",
    "battery_soc_roll_std_50",
    "cumulative_distance_m",
    "wheel_imbalance_score",
    "lidar_saturation_rate",
    "composite_health_score",
}

assert PARQUET_PATH.exists()
assert SUMMARY_PATH.exists()
assert required_output_columns.issubset(df_out.columns)
assert summary["rows_processed"] == 10_000
assert summary["rows_flagged"] >= 0
assert isinstance(summary["fields_with_anomalies_detected"], list)

print("Parquet + summary self-check passed.")


Parquet + summary self-check passed.


## 7. Stage timing and cProfile

This section identifies the slowest stage.


In [11]:
stage_times = pd.DataFrame(
    [
        {"stage": stage, "seconds": seconds}
        for stage, seconds in result.stage_times_seconds.items()
    ]
).sort_values("seconds", ascending=False)

slowest_stage = stage_times.iloc[0]["stage"]
slowest_seconds = stage_times.iloc[0]["seconds"]

print(f"Slowest stage: {slowest_stage} ({slowest_seconds:.4f} seconds)")
stage_times


Slowest stage: output (2.6214 seconds)


,stage,seconds
3,output,2.621429
1,clean,0.181982
0,ingest,0.021196
2,features,0.013153


In [12]:
profile_text = profile_pipeline(
    input_csv=DATA_PATH,
    output_parquet=OUTPUT_DIR / "profile_cleaned_features.parquet",
    summary_json=OUTPUT_DIR / "profile_validation_summary.json",
)

print(profile_text[:3000])


         1805384 function calls (1804467 primitive calls) in 0.943 seconds

   Ordered by: cumulative time
   List reduced from 1192 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.001    0.001    1.002    1.002 c:\ingen\ingen_pydev\pipeline\run_pipeline.py:31(run_pipeline)
        7    0.077    0.011    0.705    0.101 c:\ingen\ingen_pydev\pipeline\clean.py:83(_sliding_window_spike_flags)
        1    0.000    0.000    0.698    0.698 c:\ingen\ingen_pydev\pipeline\clean.py:22(clean_telemetry)
        1    0.000    0.000    0.698    0.698 c:\ingen\ingen_pydev\pipeline\clean.py:69(_flag_noise_spikes)
    70000    0.230    0.000    0.657    0.000 c:\ingen\ingen_pydev\algo\sliding_window.py:37(add)
   140000    0.136    0.000    0.248    0.000 c:\ingen\ingen_pydev\algo\sliding_window.py:71(std)
        3    0.003    0.001    0.241    0.080 c:\Users\32408\AppData\Local\Python\pythoncore-3.14-64\Lib\asyncio\base_events.py:1

## Demo conclusion

This notebook verifies the Week 3 pipeline at the assignment level:

- generated a 10,000-row synthetic telemetry CSV  
- ran the full ingest → clean → feature → output pipeline  
- produced Parquet and validation JSON outputs  
- checked GPS dropout behavior  
- checked rolling statistics against pandas ground truth  
- checked Parquet round-trip readability  
- reported runtime and slowest stage  

This is a public-safe telemetry pipeline demo. It uses synthetic data and does not depend on private robot logs, private APIs, or internal production systems.
